In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from lite_org import LITE
from utils import load_data, preprocess_data
import os

In [3]:
def plot_conv_filters(filters, colors, dataset, file_name, extension='png'):
    plt.figure(figsize=(8,8))
    for i, color in enumerate(colors):
        plt.scatter(filters[i, 0], filters[i, 1], color=color, alpha=0.5)
    plt.title('t-SNE vis. of filter')
    plt.xlabel('Component 1')
    plt.xlabel('Component 2')
    if not os.path.isdir(f'{os.getcwd()}/results_filter/{dataset}'):
        os.makedirs(f'{os.getcwd()}/results_filter/{dataset}')
    plt.savefig(f'{os.getcwd()}/results_filter/{dataset}/{file_name}.{extension}')
    plt.close()

In [4]:
dataset_names = [ 
                    'ACSF1', 'Adiac', 'AllGestureWiimoteX', 'AllGestureWiimoteY', 'AllGestureWiimoteZ',
                    'ArrowHead', 'BeetleFly', 'BirdChicken', 'Beef',  'BME', 'Car', 'CBF', 'Chinatown',
                    'ChlorineConcentration', 'CinCECGTorso', 'Coffee', 'Computers', 'CricketX',
                    'CricketY', 'CricketZ', 
                    'DiatomSizeReduction',
                    'DistalPhalanxOutlineAgeGroup', 'DistalPhalanxOutlineCorrect', 'DistalPhalanxTW',
                    'DodgerLoopDay', 'DodgerLoopGame', 'DodgerLoopWeekend', 'Earthquakes', 'ECG200',
                    'ECG5000', 'ECGFiveDays',  'EOGHorizontalSignal', 'EthanolLevel', 
                    'EOGVerticalSignal',  'FaceAll', 'FaceFour', 'FacesUCR',
                    'FiftyWords', 'Fish', 'FreezerRegularTrain',
                    'FreezerSmallTrain', 'Fungi', 'GestureMidAirD1', 'GestureMidAirD2',
                    'GestureMidAirD3', 'GesturePebbleZ1', 'GesturePebbleZ2', 'GunPoint',
                    'GunPointAgeSpan', 'GunPointMaleVersusFemale', 'GunPointOldVersusYoung',
                    'Ham',  'Haptics', 'Herring', 'HouseTwenty', 'InlineSkate',
                    'InsectEPGRegularTrain', 'InsectEPGSmallTrain', 'InsectWingbeatSound',
                    'ItalyPowerDemand', 'LargeKitchenAppliances', 'Lightning2', 'Lightning7',
                    'Mallat', 'Meat', 'MedicalImages', 'MelbournePedestrian',
                    'MiddlePhalanxOutlineAgeGroup', 'MiddlePhalanxOutlineCorrect',
                    'MiddlePhalanxTW', 'MixedShapesRegularTrain', 'MixedShapesSmallTrain',
                    'MoteStrain', 'NonInvasiveFetalECGThorax1', 'NonInvasiveFetalECGThorax2',
                    'OliveOil', 'OSULeaf', 'PhalangesOutlinesCorrect', 'Phoneme',
                    'PickupGestureWiimoteZ', 'PigAirwayPressure', 'PigArtPressure', 'PigCVP',
                    'PLAID', 'Plane', 'PowerCons', 'ProximalPhalanxOutlineAgeGroup',
                    'ProximalPhalanxOutlineCorrect', 'ProximalPhalanxTW', 'RefrigerationDevices',
                    'Rock', 'ScreenType', 'SemgHandGenderCh2', 'SemgHandMovementCh2',
                    'SemgHandSubjectCh2', 'ShakeGestureWiimoteZ', 'ShapeletSim', 'ShapesAll',
                    'SmallKitchenAppliances', 'SmoothSubspace', 'SonyAIBORobotSurface1',
                    'SonyAIBORobotSurface2', 'StarLightCurves', 'Strawberry', 'SwedishLeaf',
                    'Symbols', 'SyntheticControl', 'ToeSegmentation1', 'ToeSegmentation2', 'Trace',
                    'TwoLeadECG', 'TwoPatterns', 'UMD', 'UWaveGestureLibraryAll',
                    'UWaveGestureLibraryX', 'UWaveGestureLibraryY', 'Wine', 'WordSynonyms', 'Yoga', 
                    'UWaveGestureLibraryZ', 'Wafer',  'Worms', 'WormsTwoClass',                                         
                    'Crop', 'ElectricDevices',  'FordA', 'FordB', 'HandOutlines',                    
                    ]

dataset_names = ['SyntheticControl']
path = os.getcwd()

seed = 9409
 
for dataset in dataset_names:
    feat_path =path + f'/results_feat_map/{dataset}/'
    print('Dataset: ', dataset)
    xtrain, ytrain, xtest, ytest = load_data(dataset)
    
    trainloader = preprocess_data(xtrain, ytrain, shuffle=False)
    testloader = preprocess_data(xtest, ytest, shuffle=False)
    
    length_TS = xtrain.shape[-1]
    n_classes = len(np.unique(ytrain))
     
    model_base = LITE(output_directory=None, length_TS=length_TS, n_classes=n_classes, n_filters=[[32, 32, 32], 32, 32])
    model_base.load_state_dict(torch.load(f'/home/jabdullayev/phd/projects/CoTrain_Final_IJCNN_houma/results_oldd/base_seed_{seed}/{dataset}/best_model.pt'))


    model_dsp = torch.load(os.getcwd() + f'/results/scratch_training_1e-5_weights_4_2_1_epochs_1500/ins_sparse_seed_{seed}/{dataset}/best_model.pth')
    
    
    model_base_f_l_1 = model_base.inception.inception_layers[0].weight.squeeze(1).detach().cpu()
    model_dsp_f_l_1 = model_dsp.inception.inception_layers[0].weight.squeeze(1).detach().cpu()
    model_base_f_l_2 = model_base.inception.inception_layers[1].weight.squeeze(1).detach().cpu()
    model_dsp_f_l_2 = model_dsp.inception.inception_layers[1].weight.squeeze(1).detach().cpu()
    model_base_f_l_3 = model_base.inception.inception_layers[2].weight.squeeze(1).detach().cpu()
    model_dsp_f_l_3 = model_dsp.inception.inception_layers[2].weight.squeeze(1).detach().cpu()

    model_base_f_l_2_padded = torch.nn.functional.pad(model_base_f_l_2, (0, 20))  
    model_base_f_l_3_padded = torch.nn.functional.pad(model_base_f_l_3, (0, 30))  
    model_base_f = torch.cat([ model_base_f_l_1, model_base_f_l_2_padded, model_base_f_l_3_padded], dim=0)

    model_dsp_f_l_2_padded = torch.nn.functional.pad(model_dsp_f_l_2, (0, 20))  
    model_dsp_f_l_3_padded = torch.nn.functional.pad(model_dsp_f_l_3, (0, 30))  
    model_dsp_f = torch.cat([ model_dsp_f_l_1, model_dsp_f_l_2_padded, model_dsp_f_l_3_padded], dim=0)


    print(f'model_dsp_f_l_1 {model_dsp_f_l_1.shape}')
    print(f'model_dsp_f_l_2 {model_dsp_f_l_2.shape}')
    print(f'model_dsp_f_l_3 {model_dsp_f_l_3.shape}')
    
    from dtaidistance import dtw

    filters = torch.cat([model_base_f, model_dsp_f], dim=0)
    filters = filters.detach().cpu().numpy()
    from sklearn.preprocessing import StandardScaler

    X_std = StandardScaler().fit_transform(filters)

    from sklearn.manifold import TSNE
    ds = dtw.distance_matrix_fast(X_std.astype(np.double))

    tsne = TSNE(n_components=2, metric='precomputed', init='random', perplexity=15, random_state=42)
    filters_2d = tsne.fit_transform(ds)

    colors = ['red'] * len(model_base_f) + ['blue'] * len(model_dsp_f) 
    
    plot_conv_filters(filters_2d, colors, dataset, 'base')
    
    
    filters_co = torch.cat([model_base_f, model_dsp_f], dim=0)
    filters_co = filters_co.detach().cpu().numpy()
    from sklearn.preprocessing import StandardScaler

    X_std_co = StandardScaler().fit_transform(filters_co)

    from sklearn.manifold import TSNE
    ds_co = dtw.distance_matrix_fast(X_std_co.astype(np.double))

    tsne_co = TSNE(n_components=2, metric='precomputed', init='random', perplexity=15, random_state=42)
    filters_2d_co = tsne_co.fit_transform(ds_co)

    colors = ['red'] * len(model_base_f) + ['blue'] * len(model_dsp_f)

    plot_conv_filters(filters_2d_co, colors, dataset, 'co', extension='png')

Dataset:  SyntheticControl
model_dsp_f_l_1 torch.Size([15, 40])
model_dsp_f_l_2 torch.Size([8, 20])
model_dsp_f_l_3 torch.Size([2, 10])


/tmp/ipykernel_366181/1201721547.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_base.load_state_dict(torch.load(f'/home/jabdullayev/phd/projects/CoTrain_Final_IJ

In [19]:
model_dsp.inception.inception_layers[1].weight.shape

torch.Size([8, 1, 20])

In [5]:
model_base_f.shape

torch.Size([96, 40])